# OpenCV in Practice, M2: find every ball on a pool table

Every code block in the posts, in order, so you can run them on your own machine. On the
site they run in the browser; here they run on the OpenCV you install below.

Photos "Billiards table 2.JPG", "Billiards table 1.JPG" and "Billiards Table.JPG" by MarkBuckawicki (Wikimedia Commons, CC0), resized by this repository's `practice-m2`; ball positions placed by hand.

In [ ]:
# Pinned so a Colab base-image change cannot silently move the numbers.
%pip install -q 'opencv-python>=5.0.0,<6.0.0' 'numpy>=2' 'matplotlib>=3.8'

In [ ]:
import os
import urllib.request

import cv2
import matplotlib.pyplot as plt

print("OpenCV", cv2.__version__)

# The module's images, served by condados.ai. Cloudflare refuses Python's
# default user agent, so the request names itself.
BASE = os.environ.get("PRACTICE_BASE", "https://condados.ai")
UA = {"User-Agent": "fundamentals-lab/0.1 (+https://github.com/CondadosAI/fundamentals-lab)"}
for path in ['/practice/m2/pool.webp', '/practice/m2/pool-close.webp', '/practice/m2/pool-wide.webp', '/practice/m2/labels.json']:
    name = path.rsplit("/", 1)[-1]
    if not os.path.exists(name):
        req = urllib.request.Request(BASE + path, headers=UA)
        open(name, "wb").write(urllib.request.urlopen(req).read())
print("images:", 'pool.webp', 'pool-close.webp', 'pool-wide.webp', 'labels.json')

In [ ]:
# cv2.imshow opens a window in a script and does nothing useful in a notebook.
# Draw with matplotlib instead (OpenCV is BGR, matplotlib expects RGB).
def _imshow(title, img):
    plt.figure(figsize=(10, 6))
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.title(title)
    plt.axis("off")
    plt.show()

cv2.imshow = _imshow
cv2.waitKey = lambda delay=0: -1

## Find the HSV Range of an Object in OpenCV

https://condados.ai/blog/opencv-find-hsv-range

In [ ]:
import cv2
import numpy as np

img = cv2.imread("pool.webp")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# (x, y) of four points on the photo
points = {"cloth, near": (400, 240),
          "cloth, far": (700, 95),
          "shadow": (790, 336),
          "green ball": (727, 155)}
for name, (x, y) in points.items():
    print(f"{name:12} BGR {img[y, x]}  HSV {hsv[y, x]}")

In [ ]:
# a patch of the photo that is only cloth
patch = hsv[550:, 650:]
for i, name in enumerate(["H", "S", "V"]):
    lo, hi = np.percentile(patch[..., i], [2, 98])
    print(f"{name}: {lo:.0f} to {hi:.0f}")
cv2.imshow("hue (0-179)", hsv[..., 0])

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# RGB2HSV on a BGR image: red and blue swapped
wrong = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
x, y = points["cloth, near"]
print("right:", hsv[y, x], " wrong:", wrong[y, x])
cv2.imshow("hue, wrong flag", wrong[..., 0])

**Your turn.** Index hsv with [y, x], then take the first of the three numbers with [0].

In [ ]:
hue = None  # replace this line
print(hue)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(125 <= hue <= 140)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Make a Color Mask with cv2.inRange in OpenCV

https://condados.ai/blog/opencv-inrange-color-mask

In [ ]:
import cv2
import numpy as np

img = cv2.imread("pool.webp")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

lower = np.array([75, 80, 40])
upper = np.array([100, 255, 255])
cloth = cv2.inRange(hsv, lower, upper)

print(cloth.shape, cloth.dtype, np.unique(cloth))
print("cloth:", round(100 * np.mean(cloth > 0), 1), "% of pixels")
cv2.imshow("cloth", cloth)

In [ ]:
not_cloth = cv2.bitwise_not(cloth)
# keep the photo's colours where the mask is white
cut = cv2.bitwise_and(img, img, mask=not_cloth)
cv2.imshow("not cloth", cut)

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# the mask passed as the second image, not as mask=
cut = cv2.bitwise_and(img, not_cloth)

**Your turn.** Let hue run over its whole range, 0 to 179. Try saturation from 0 to 40 and value from 200 to 255.

In [ ]:
white = cv2.inRange(hsv, (0, 0, 0), (179, 255, 255))  # narrow this
cv2.imshow("white", white)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(white[311, 791] == 255 and white[500, 100] == 0)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Erode, Dilate, Open and Close a Mask in OpenCV

https://condados.ai/blog/opencv-morphology-erode-dilate

In [ ]:
import cv2
import numpy as np

img = cv2.imread("pool.webp")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
cloth = cv2.inRange(hsv, (75, 80, 40), (100, 255, 255))
# the table's outline: lesson 4 explains these three lines
cs, _ = cv2.findContours(cloth, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
table = np.zeros_like(cloth)
cv2.fillPoly(table, [cv2.convexHull(max(cs, key=cv2.contourArea))], 255)
objects = cv2.bitwise_and(cv2.bitwise_not(cloth), table)

def blobs(mask):
    return cv2.connectedComponents(mask)[0] - 1
print("blobs:", blobs(objects))

In [ ]:
kernel = np.ones((5, 5), np.uint8)
eroded = cv2.erode(objects, kernel)
dilated = cv2.dilate(objects, kernel)
print("eroded:", blobs(eroded), " dilated:", blobs(dilated))

In [ ]:
opened = cv2.morphologyEx(objects, cv2.MORPH_OPEN, kernel)
k7 = np.ones((7, 7), np.uint8)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, k7)
print("opened:", blobs(opened), " then closed:", blobs(closed))
cv2.imshow("opened, then closed", closed)

In [ ]:
table = cv2.erode(table, np.ones((19, 19), np.uint8))
objects = cv2.bitwise_and(cv2.bitwise_not(cloth), table)
opened = cv2.morphologyEx(objects, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, k7)
print("blobs:", blobs(closed))
cv2.imshow("with the table pulled in", closed)

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
big = np.ones((25, 25), np.uint8)
too_much = cv2.morphologyEx(objects, cv2.MORPH_OPEN, big)
print("blobs:", blobs(too_much))
cv2.imshow("opened with 25 x 25", too_much)

**Your turn.** Try sizes between 11 and 25. The ball is about 26 pixels across, and the kernel has to fit inside it.

In [ ]:
size = 5  # change this
k = np.ones((size, size), np.uint8)
opened = cv2.morphologyEx(objects, cv2.MORPH_OPEN, k)
print(blobs(opened), opened[92, 494])

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(opened[92, 494] == 255 and size >= 17)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Find, Measure and Filter Contours in OpenCV

https://condados.ai/blog/opencv-find-contours

In [ ]:
import cv2
import numpy as np

img = cv2.imread("pool.webp")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
cloth = cv2.inRange(hsv, (75, 80, 40), (100, 255, 255))
cs, _ = cv2.findContours(cloth, cv2.RETR_EXTERNAL,
                         cv2.CHAIN_APPROX_SIMPLE)
biggest = max(cs, key=cv2.contourArea)
hull = cv2.convexHull(biggest)
print(len(cs), "contours; biggest", cv2.contourArea(biggest))
print("hull corners:", len(hull))

In [ ]:
table = np.zeros_like(cloth)
cv2.fillPoly(table, [hull], 255)
table = cv2.erode(table, np.ones((19, 19), np.uint8))
objects = cv2.bitwise_and(cv2.bitwise_not(cloth), table)
k5, k7 = np.ones((5, 5), np.uint8), np.ones((7, 7), np.uint8)
objects = cv2.morphologyEx(objects, cv2.MORPH_OPEN, k5)
objects = cv2.morphologyEx(objects, cv2.MORPH_CLOSE, k7)
cs, _ = cv2.findContours(objects, cv2.RETR_EXTERNAL,
                         cv2.CHAIN_APPROX_SIMPLE)
print(len(cs), "contours on the table")

In [ ]:
def roundness(c):
    # 1.0 for a circle, near 0 for a stick
    area = cv2.contourArea(c)
    return 4 * np.pi * area / cv2.arcLength(c, True) ** 2

for c in sorted(cs, key=cv2.contourArea, reverse=True):
    a = cv2.contourArea(c)
    print(f"area {a:6.0f}   roundness {roundness(c):.2f}")

In [ ]:
out = img.copy()
balls = []
for c in cs:
    if cv2.contourArea(c) >= 45 and roundness(c) > 0.55:
        (x, y), r = cv2.minEnclosingCircle(c)
        balls.append((round(x), round(y)))
        cv2.circle(out, (int(x), int(y)), int(r) + 4, (0, 255, 0), 2)
print(len(balls), "balls:", balls)
cv2.imshow("round blobs", out)

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
big = [c for c in cs if cv2.contourArea(c) >= 45]
print("kept by area alone:", len(big))
cv2.imshow("all blobs", cv2.drawContours(img.copy(), big, -1, (0, 0, 255), 2))

**Your turn.** max(cs, key=cv2.contourArea) gives the largest contour; pass it to roundness().

In [ ]:
r = None  # replace this line
print(r)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(r < 0.2)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Where Color Detection Fails in OpenCV

https://condados.ai/blog/opencv-color-detection-failures

In [ ]:
import cv2
import numpy as np

def objects_on_table(hsv, v_min):
    cloth = cv2.inRange(hsv, (75, 80, v_min), (100, 255, 255))
    cs, _ = cv2.findContours(cloth, cv2.RETR_EXTERNAL,
                             cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(max(cs, key=cv2.contourArea))
    table = cv2.fillPoly(np.zeros_like(cloth), [hull], 255)
    table = cv2.erode(table, np.ones((19, 19), np.uint8))
    obj = cv2.bitwise_and(cv2.bitwise_not(cloth), table)
    k5, k7 = np.ones((5, 5), np.uint8), np.ones((7, 7), np.uint8)
    obj = cv2.morphologyEx(obj, cv2.MORPH_OPEN, k5)
    return cv2.morphologyEx(obj, cv2.MORPH_CLOSE, k7)

In [ ]:
def find_balls(img, v_min=40, pocket_v=0):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    obj = objects_on_table(hsv, v_min)
    cs, _ = cv2.findContours(obj, cv2.RETR_EXTERNAL,
                             cv2.CHAIN_APPROX_SIMPLE)
    balls = []
    for c in cs:
        a, p = cv2.contourArea(c), cv2.arcLength(c, True)
        blob = cv2.drawContours(np.zeros_like(obj), [c], -1, 255, -1)
        brightest = hsv[..., 2][blob > 0].max()
        round_ = a >= 45 and 4 * np.pi * a / p**2 > 0.55
        if round_ and brightest >= pocket_v:
            balls.append(cv2.minEnclosingCircle(c)[0])
    return balls

In [ ]:
import json
labels = json.load(open("labels.json"))

def score(balls, truth):
    # closest pairs first; a hit lands inside the ball
    pairs = sorted((np.hypot(bx - x, by - y), i, j)
                   for i, (x, y, r) in enumerate(truth)
                   for j, (bx, by) in enumerate(balls))
    hit_t, hit_b = set(), set()
    for d, i, j in pairs:
        if i not in hit_t and j not in hit_b and d < max(truth[i][2], 8):
            hit_t.add(i)
            hit_b.add(j)
    return len(hit_t), len(balls) - len(hit_b)

In [ ]:
photos = ["pool", "pool-close", "pool-wide"]
imgs = {n: cv2.imread(f"{n}.webp") for n in photos}

def report(**settings):
    total = np.zeros(2, int)
    for n in photos:
        found, false = score(find_balls(imgs[n], **settings), labels[n])
        total += (found, false)
        print(f"{n:11} {found:2} of {len(labels[n])}, {false} false")
    print(f"{'total':11} {total[0]} of 41, {total[1]} false")

report()

In [ ]:
report(pocket_v=60)

In [ ]:
report(v_min=10, pocket_v=60)

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
report(v_min=0)

**Your turn.** Loop over photos, call score(find_balls(imgs[n], v_min=20, pocket_v=60), labels[n]) and add up the first number it returns.

In [ ]:
found20 = 0
# your loop here
print(found20)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(found20 == 28)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Find Every Ball on a Pool Table with OpenCV

https://condados.ai/blog/find-balls-pool-table-opencv

In [ ]:
import cv2
import numpy as np

def objects_on_table(hsv, v_min):
    cloth = cv2.inRange(hsv, (75, 80, v_min), (100, 255, 255))
    cs, _ = cv2.findContours(cloth, cv2.RETR_EXTERNAL,
                             cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(max(cs, key=cv2.contourArea))
    table = cv2.fillPoly(np.zeros_like(cloth), [hull], 255)
    table = cv2.erode(table, np.ones((19, 19), np.uint8))
    obj = cv2.bitwise_and(cv2.bitwise_not(cloth), table)
    k5, k7 = np.ones((5, 5), np.uint8), np.ones((7, 7), np.uint8)
    obj = cv2.morphologyEx(obj, cv2.MORPH_OPEN, k5)
    return cv2.morphologyEx(obj, cv2.MORPH_CLOSE, k7)

In [ ]:
def find_balls(img, v_min=10, pocket_v=60):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    obj = objects_on_table(hsv, v_min)
    cs, _ = cv2.findContours(obj, cv2.RETR_EXTERNAL,
                             cv2.CHAIN_APPROX_SIMPLE)
    balls = []
    for c in cs:
        a, p = cv2.contourArea(c), cv2.arcLength(c, True)
        blob = cv2.drawContours(np.zeros_like(obj), [c], -1, 255, -1)
        brightest = hsv[..., 2][blob > 0].max()
        round_ = a >= 45 and 4 * np.pi * a / p**2 > 0.55
        if round_ and brightest >= pocket_v:
            balls.append(cv2.minEnclosingCircle(c)[0])
    return balls

In [ ]:
import json
labels = json.load(open("labels.json"))

def score(balls, truth):
    pairs = sorted((np.hypot(bx - x, by - y), i, j)
                   for i, (x, y, r) in enumerate(truth)
                   for j, (bx, by) in enumerate(balls))
    hit_t, hit_b = set(), set()
    for d, i, j in pairs:
        if i not in hit_t and j not in hit_b and d < max(truth[i][2], 8):
            hit_t.add(i)
            hit_b.add(j)
    return len(hit_t), len(balls) - len(hit_b)

for n in ["pool", "pool-close", "pool-wide"]:
    found, false = score(find_balls(cv2.imread(f"{n}.webp")), labels[n])
    print(f"{n:11} {found:2} of {len(labels[n])}, {false} false")

In [ ]:
out = cv2.imread("pool.webp")
for x, y in find_balls(out):
    cv2.circle(out, (int(x), int(y)), 24, (0, 255, 0), 3)
cv2.imshow("pool.webp: what the pipeline kept", out)